In [0]:
%sql
ALTER TABLE workspace.silver.web_events_clean ALTER COLUMN event_id SET NOT NULL;
ALTER TABLE workspace.silver.web_events_clean ALTER COLUMN custkey SET NOT NULL;
ALTER TABLE workspace.silver.web_events_clean
  ADD CONSTRAINT valid_event_type CHECK (event_type IN ('page_view', 'purchase', 'refund'));

In [0]:
%sql
INSERT INTO workspace.silver.web_events_clean (event_id, custkey, event_type, amount, event_time)
VALUES (99, 123, 'hack', 10.00, current_timestamp());

In [0]:
%python
bad = spark.createDataFrame([(1, "x")], ["foo", "bar"])
bad.write.mode("append").saveAsTable("workspace.silver.web_events_clean")

In [0]:
%sql
SHOW TBLPROPERTIES workspace.silver.web_events_clean;

In [0]:
%python
import os, sys
sys.path.append(os.path.dirname(os.getcwd()))

from src.transform import clean_web_events, get_rejected_events

bronze = spark.table("workspace.bronze.web_events")
print(clean_web_events(bronze).count())
print(get_rejected_events(bronze).count())

In [0]:
%python
import os, sys
sys.path.append(os.path.dirname(os.getcwd()))

from src.validate import check_schema, check_no_nulls, check_unique, check_allowed_values

silver = spark.table("workspace.silver.web_events_clean")

check_schema(silver, {"event_id": "bigint", "custkey": "bigint", "event_type": "string",
                      "amount": "decimal(12,2)", "event_time": "timestamp"})
check_no_nulls(silver, ["event_id", "custkey"])
check_unique(silver, "event_id")
check_allowed_values(silver, "event_type", ["page_view", "purchase", "refund"])

print("All Silver checks passed")

In [0]:
%python
drifted = silver.withColumnRenamed("amount", "amt")
check_schema(drifted, {"amount": "decimal(10,2)"})